# Data Collection & Cleaning

Combines four quarterly BTS DB1B Market files (Q3 2024 – Q2 2025), removes incomplete records, and joins airport lat/lon from OurAirports for use as GNN node features.

In [ ]:
import pandas as pd
import numpy as np
import os

## 1. Preview Raw Data

In [ ]:
sample = pd.read_csv('../../data/rawdata/2024_q3.csv', low_memory=False)
print(f"Shape: {sample.shape}")
print(f"Columns: {list(sample.columns)}")
sample.head()

Shape: (8297869, 15)
Columns: ['YEAR', 'QUARTER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST', 'REPORTING_CARRIER', 'TICKET_CARRIER', 'OPERATING_CARRIER', 'BULK_FARE', 'PASSENGERS', 'MARKET_FARE', 'MARKET_DISTANCE', 'NONSTOP_MILES', 'MKT_GEO_TYPE']


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2


## 2. Combine Quarterly Files

In [ ]:
files = [
    '../../data/rawdata/2024_q3.csv',
    '../../data/rawdata/2024_q4.csv',
    '../../data/rawdata/2025_q1.csv',
    '../../data/rawdata/2025_q2.csv',
]

dfs = []
for path in files:
    df = pd.read_csv(path, low_memory=False)
    print(f"  {os.path.basename(path)}: {df.shape}")
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
print(f"\nCombined shape: {combined.shape}")
print(f"Quarters present: {sorted(combined[['YEAR','QUARTER']].drop_duplicates().apply(tuple, axis=1).tolist())}")
combined.head()

  2024_q3.csv: (8297869, 15)
  2024_q4.csv: (8525077, 15)
  2025_q1.csv: (7297028, 15)
  2025_q2.csv: (8450420, 15)

Combined shape: (32570394, 15)
Quarters present: [(2024, 3), (2024, 4), (2025, 1), (2025, 2)]


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2


## 3. Missing Value Analysis

In [ ]:
print("=== Missing Value Counts ===")
print(combined.isna().sum())
print(f"\nRows with any missing value: {combined.isna().any(axis=1).sum():,}  ({combined.isna().any(axis=1).mean():.1%} of total)")

=== Missing Value Counts ===
YEAR                 0
QUARTER              0
ORIGIN_AIRPORT_ID    0
ORIGIN               0
DEST_AIRPORT_ID      0
DEST                 0
REPORTING_CARRIER    0
TICKET_CARRIER       0
OPERATING_CARRIER    0
BULK_FARE            0
PASSENGERS           0
MARKET_FARE          0
MARKET_DISTANCE      0
NONSTOP_MILES        0
MKT_GEO_TYPE         0
dtype: int64

Rows with any missing value: 0  (0.0% of total)


## 4. Clean Data

Drop rows with missing values and remove zero/negative fares (data entry artifacts).

In [ ]:
df_clean = (
    combined
    .dropna()
    .query("MARKET_FARE > 0")
    .reset_index(drop=True)
)

print(f"Before cleaning: {combined.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Removed:         {combined.shape[0] - df_clean.shape[0]:,} rows ({(combined.shape[0] - df_clean.shape[0]) / combined.shape[0]:.1%})")
df_clean.describe()

Before cleaning: 32,570,394 rows
After cleaning:  32,541,848 rows
Removed:         28,546 rows (0.1%)


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
count,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07,32541848.0,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07
mean,2.024483e+03,2.554236e+00,1.282923e+04,1.282866e+04,0.0,1.895133e+00,2.710958e+02,1.301878e+03,1.221670e+03,1.932522e+00
std,4.997272e-01,1.103920e+00,1.541862e+03,1.542257e+03,0.0,4.971489e+00,2.216202e+02,8.166206e+02,7.736060e+02,2.508474e-01
min,2.024000e+03,1.000000e+00,1.013500e+04,1.013500e+04,0.0,1.000000e+00,1.100000e-01,1.100000e+01,1.100000e+01,1.000000e+00
25%,2.024000e+03,2.000000e+00,1.129800e+04,1.129800e+04,0.0,1.000000e+00,1.490000e+02,7.100000e+02,6.540000e+02,2.000000e+00
50%,2.024000e+03,3.000000e+00,1.294500e+04,1.294500e+04,0.0,1.000000e+00,2.310000e+02,1.087000e+03,1.020000e+03,2.000000e+00
75%,2.025000e+03,4.000000e+00,1.410700e+04,1.410700e+04,0.0,1.000000e+00,3.400000e+02,1.751000e+03,1.643000e+03,2.000000e+00
max,2.025000e+03,4.000000e+00,1.686900e+04,1.686900e+04,0.0,1.032000e+03,4.443200e+04,1.134400e+04,9.411000e+03,2.000000e+00


# realized that there are still abnormally low fares
Seached on google and various sites and found the cheapest flight is $19 from pittsburg to dubois, so I am going to filter that out now

In [ ]:

df_clean[df_clean['MARKET_FARE'] < 40]

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.50,1961.0,1738.0,2
39,2024,3,10135,ABE,10157,ACV,G7,UA,99,0.0,1.0,5.00,2976.0,2518.0,2
40,2024,3,10135,ABE,10157,ACV,OO,99,99,0.0,1.0,5.38,2698.0,2518.0,2
75,2024,3,10135,ABE,10397,ATL,9E,DL,9E,0.0,1.0,2.86,692.0,692.0,2
76,2024,3,10135,ABE,10397,ATL,9E,DL,9E,0.0,1.0,5.50,692.0,692.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32541567,2025,2,16869,XWA,14747,SEA,OO,UA,99,0.0,1.0,5.00,1606.0,863.0,2
32541683,2025,2,16869,XWA,14869,SLC,UA,UA,99,0.0,1.0,5.50,973.0,656.0,2
32541726,2025,2,16869,XWA,15016,STL,OO,DL,99,0.0,1.0,5.50,1001.0,937.0,2
32541776,2025,2,16869,XWA,15370,TUL,OO,UA,OO,0.0,1.0,5.50,1123.0,924.0,2


In [ ]:
df_clean = (
    combined
    .dropna()
    .query("MARKET_FARE > 19")
    .reset_index(drop=True)
)

print(f"Before cleaning: {combined.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Removed:         {combined.shape[0] - df_clean.shape[0]:,} rows ({(combined.shape[0] - df_clean.shape[0]) / combined.shape[0]:.1%})")
df_clean.describe()

Before cleaning: 32,570,394 rows
After cleaning:  30,976,039 rows
Removed:         1,594,355 rows (4.9%)


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
count,3.097604e+07,3.097604e+07,3.097604e+07,3.097604e+07,30976039.0,3.097604e+07,3.097604e+07,3.097604e+07,3.097604e+07,3.097604e+07
mean,2.024483e+03,2.556007e+00,1.282755e+04,1.282683e+04,0.0,1.800310e+00,2.845049e+02,1.294039e+03,1.216806e+03,1.932594e+00
std,4.996969e-01,1.103854e+00,1.540203e+03,1.540644e+03,0.0,3.411450e+00,2.187720e+02,8.137536e+02,7.715996e+02,2.507233e-01
min,2.024000e+03,1.000000e+00,1.013500e+04,1.013500e+04,0.0,1.000000e+00,1.901000e+01,1.100000e+01,1.100000e+01,1.000000e+00
25%,2.024000e+03,2.000000e+00,1.129800e+04,1.129800e+04,0.0,1.000000e+00,1.615000e+02,7.010000e+02,6.510000e+02,2.000000e+00
50%,2.024000e+03,3.000000e+00,1.291500e+04,1.291500e+04,0.0,1.000000e+00,2.390000e+02,1.080000e+03,1.015000e+03,2.000000e+00
75%,2.025000e+03,4.000000e+00,1.410700e+04,1.410700e+04,0.0,1.000000e+00,3.475000e+02,1.744000e+03,1.633000e+03,2.000000e+00
max,2.025000e+03,4.000000e+00,1.686900e+04,1.686900e+04,0.0,4.930000e+02,4.443200e+04,1.134400e+04,9.362000e+03,2.000000e+00


In [ ]:
#checking distribution now
df_clean['MARKET_FARE'].value_counts().sort_index()

MARKET_FARE
19.01       14
19.02       10
19.03       11
19.04       20
19.05        5
            ..
19653.00     1
23180.00     1
30656.00     1
38735.00     1
44432.00     1
Name: count, Length: 151785, dtype: int64

In [ ]:
#checking if these low price flights make sense 
df_clean[df_clean['MARKET_FARE'] < 40].sort_values('MARKET_DISTANCE', ascending=False).head(20)

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
14932322,2024,4,14747,SEA,12016,GUM,UA,UA,UA,0.0,1.0,35.0,9579.0,5674.0,1
18438108,2025,1,12016,GUM,10721,BOS,UA,UA,UA,0.0,1.0,38.0,9302.0,7967.0,1
14336768,2024,4,14492,RDU,12016,GUM,UA,UA,UA,0.0,1.0,26.0,8842.0,8004.0,1
2883350,2024,3,12016,GUM,13577,MYR,UA,UA,99,0.0,1.0,39.0,8798.0,8098.0,1
21454547,2025,1,14492,RDU,12016,GUM,UA,UA,UA,0.0,1.0,22.0,8747.0,8004.0,1
25804464,2025,2,12016,GUM,10821,BWI,UA,UA,UA,0.0,1.0,35.0,8657.0,7932.0,1
10811304,2024,4,12016,GUM,10397,ATL,UA,UA,UA,0.0,1.0,34.0,8650.0,7855.0,1
10811396,2024,4,12016,GUM,11278,DCA,UA,UA,UA,0.0,1.0,36.0,8642.0,7934.0,1
19798458,2025,1,13204,MCO,12016,GUM,UA,UA,UA,0.0,1.0,34.0,8559.0,8216.0,1
18438840,2025,1,12016,GUM,14730,SDF,UA,UA,99,0.0,1.0,33.0,8493.0,7609.0,1


## 5. Join Airport Coordinates

`airports.dat.txt` (OurAirports) provides latitude and longitude keyed on IATA code. We join on `ORIGIN` and `DEST` to add geo features needed by the GNN node embeddings.

In [ ]:
airport_cols = ['airport_id', 'name', 'city', 'country', 'iata', 'icao',
                'latitude', 'longitude', 'altitude', 'timezone', 'dst', 'tz_db', 'type', 'source']

airports_raw = pd.read_csv('../../data/rawdata/airports.dat.txt', header=None, names=airport_cols)

# Keep only rows with a valid IATA code
coords = (
    airports_raw[airports_raw['iata'].notna() & (airports_raw['iata'] != '\\N')]
    [['iata', 'latitude', 'longitude']]
    .drop_duplicates('iata')
)

print(f"Airports with valid IATA codes: {len(coords):,}")
coords.head()

Airports with valid IATA codes: 6,072


,iata,latitude,longitude
0,GKA,-6.081690,145.391998
1,MAG,-5.207080,145.789001
2,HGU,-5.826790,144.296005
3,LAE,-6.569803,146.725977
4,POM,-9.443380,147.220001


In [ ]:
df_geo = (
    df_clean
    .merge(
        coords.rename(columns={'iata': 'ORIGIN', 'latitude': 'ORIGIN_LAT', 'longitude': 'ORIGIN_LON'}),
        on='ORIGIN', how='inner'
    )
    .merge(
        coords.rename(columns={'iata': 'DEST', 'latitude': 'DEST_LAT', 'longitude': 'DEST_LON'}),
        on='DEST', how='inner'
    )
)

print(f"After geo join: {df_geo.shape[0]:,} rows, {df_geo.shape[1]} columns")
print(f"Rows lost in join (airports not in coords): {df_clean.shape[0] - df_geo.shape[0]:,}")
df_geo.head()

After geo join: 32,520,273 rows, 19 columns
Rows lost in join (airports not in coords): 21,575


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE,ORIGIN_LAT,ORIGIN_LON,DEST_LAT,DEST_LON
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2,40.6521,-75.440804,32.411301,-99.681900
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001


In [ ]:
print("=== Final Dataset Summary ===")
print(f"Shape: {df_geo.shape}")
print(f"\nMissing values: {df_geo.isna().sum().sum()}")
print(f"\nRow counts by quarter:")
print(df_geo.groupby(['YEAR', 'QUARTER']).size().to_string())
print(f"\nUnique origin airports:  {df_geo['ORIGIN'].nunique():,}")
print(f"Unique dest airports:    {df_geo['DEST'].nunique():,}")
print(f"Unique routes:           {df_geo.groupby(['ORIGIN', 'DEST']).ngroups:,}")
print(f"\nMARKET_FARE stats:")
print(df_geo['MARKET_FARE'].describe())

=== Final Dataset Summary ===
Shape: (32520273, 19)

Missing values: 0

Row counts by quarter:
YEAR  QUARTER
2024  3          8284996
      4          8512287
2025  1          7285614
      2          8437376

Unique origin airports:  451
Unique dest airports:    453
Unique routes:           86,196

MARKET_FARE stats:
count    3.252027e+07
mean     2.710284e+02
std      2.215013e+02
min      1.100000e-01
25%      1.490000e+02
50%      2.309200e+02
75%      3.398700e+02
max      4.443200e+04
Name: MARKET_FARE, dtype: float64


## 6. Save Clean Data

In [ ]:
out_path = '../../data/clean_data/T_DB1B_MARKET_CLEAN.csv'
df_geo.to_csv(out_path, index=False)
print(f"Saved {df_geo.shape[0]:,} rows × {df_geo.shape[1]} columns → {out_path}")

Saved 32,520,273 rows × 19 columns → ../../data/clean_data/T_DB1B_MARKET_CLEAN.csv


## Preview raw data and combine raw data

In [3]:
import pandas as pd


In [6]:
preview=pd.read_csv('../../data/rawdata/T_DB1B_MARKET1.csv')
preview.head()

,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,DISTANCE_GROUP,MARKET_MILES_FLOWN,NONSTOP_MILES,MKT_GEO_TYPE
0,10135,1013506,30135,10136,1013603,30136,OH,AA,99,0.0,1.0,895.41,1575.0,4,1575.0,1458.0,2
1,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,168.50,1961.0,4,1961.0,1738.0,2
2,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,201.84,1961.0,4,1961.0,1738.0,2
3,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,361.50,1961.0,4,1961.0,1738.0,2
4,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,843.00,1961.0,4,1961.0,1738.0,2


In [7]:


# List of CSV files to combine
csv_files = [
    "../../data/rawdata/T_DB1B_MARKET1.csv",
    "../../data/rawdata/T_DB1B_MARKET2.csv",
    "../../data/rawdata/T_DB1B_MARKET3.csv",
    "../../data/rawdata/T_DB1B_MARKET4.csv"
]

# Initialize an empty list to store dataframes
dfs = []

# Read and append each CSV file
for file in csv_files:
    print(f"Reading {file}...")
    df = pd.read_csv(file)
    print(f"  Shape: {df.shape}")
    dfs.append(df)

# Combine all dataframes
print("\nCombining all files...")
combined_df = pd.concat(dfs, ignore_index=True)

print(f"\nCombined dataset shape: {combined_df.shape}")
print(f"Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Save the combined dataset
output_path = "../../data/rawdata/T_DB1B_MARKET_COMBINED.csv"
print(f"\nSaving to {output_path}...")
combined_df.to_csv(output_path, index=False)

print("Done!")

Reading ../../data/rawdata/T_DB1B_MARKET1.csv...
  Shape: (7297028, 17)
Reading ../../data/rawdata/T_DB1B_MARKET2.csv...
  Shape: (8450420, 17)
Reading ../../data/rawdata/T_DB1B_MARKET3.csv...
  Shape: (8297869, 17)
Reading ../../data/rawdata/T_DB1B_MARKET4.csv...
  Shape: (8297869, 6)

Combining all files...

Combined dataset shape: (32343186, 17)
Memory usage: 7722.82 MB

Saving to ../../data/rawdata/T_DB1B_MARKET_COMBINED.csv...
Done!


In [8]:
combined_df=pd.read_csv('../../data/rawdata/T_DB1B_MARKET_COMBINED.csv')
combined_df.head()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_48832\3461268546.py:1: DtypeWarning: Columns (6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_df=pd.read_csv('../../data/rawdata/T_DB1B_MARKET_COMBINED.csv')


,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,DISTANCE_GROUP,MARKET_MILES_FLOWN,NONSTOP_MILES,MKT_GEO_TYPE
0,10135,1013506,30135,10136,1013603,30136,OH,AA,99,0.0,1.0,895.41,1575.0,4.0,1575.0,1458.0,2.0
1,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,168.50,1961.0,4.0,1961.0,1738.0,2.0
2,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,201.84,1961.0,4.0,1961.0,1738.0,2.0
3,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,361.50,1961.0,4.0,1961.0,1738.0,2.0
4,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,843.00,1961.0,4.0,1961.0,1738.0,2.0


## Data cleaning

In [4]:
# Data Loading
pd.set_option('display.max_columns', None)

df = pd.read_csv('../../data/rawdata/T_DB1B_MARKET_COMBINED.csv')   

# Data Overview
print("Data Loaded")
print(df.shape)
df.head()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_17724\2164937600.py:4: DtypeWarning: Columns (6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../data/rawdata/T_DB1B_MARKET_COMBINED.csv')


Data Loaded
(32343186, 17)


,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,DISTANCE_GROUP,MARKET_MILES_FLOWN,NONSTOP_MILES,MKT_GEO_TYPE
0,10135,1013506,30135,10136,1013603,30136,OH,AA,99,0.0,1.0,895.41,1575.0,4.0,1575.0,1458.0,2.0
1,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,168.50,1961.0,4.0,1961.0,1738.0,2.0
2,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,201.84,1961.0,4.0,1961.0,1738.0,2.0
3,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,361.50,1961.0,4.0,1961.0,1738.0,2.0
4,10135,1013506,30135,10140,1014005,30140,9E,DL,99,0.0,1.0,843.00,1961.0,4.0,1961.0,1738.0,2.0


In [5]:
# Data Structure
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32343186 entries, 0 to 32343185
Data columns (total 17 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ORIGIN_AIRPORT_ID      int64  
 1   ORIGIN_AIRPORT_SEQ_ID  int64  
 2   ORIGIN_CITY_MARKET_ID  int64  
 3   DEST_AIRPORT_ID        int64  
 4   DEST_AIRPORT_SEQ_ID    int64  
 5   DEST_CITY_MARKET_ID    int64  
 6   REPORTING_CARRIER      object 
 7   TICKET_CARRIER         object 
 8   OPERATING_CARRIER      object 
 9   BULK_FARE              float64
 10  PASSENGERS             float64
 11  MARKET_FARE            float64
 12  MARKET_DISTANCE        float64
 13  DISTANCE_GROUP         float64
 14  MARKET_MILES_FLOWN     float64
 15  NONSTOP_MILES          float64
 16  MKT_GEO_TYPE           float64
dtypes: float64(8), int64(6), object(3)
memory usage: 4.1+ GB


,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,DISTANCE_GROUP,MARKET_MILES_FLOWN,NONSTOP_MILES,MKT_GEO_TYPE
count,3.234319e+07,3.234319e+07,3.234319e+07,3.234319e+07,3.234319e+07,3.234319e+07,2.404532e+07,2.404532e+07,2.404532e+07,2.404532e+07,2.404532e+07,2.404532e+07,2.404532e+07,2.404532e+07
mean,1.282563e+04,1.282567e+06,3.196636e+04,1.282417e+04,1.282421e+06,3.196489e+04,8.442392e-05,1.904901e+00,2.684043e+02,1.308387e+03,3.114319e+00,1.305384e+03,1.227264e+03,1.930577e+00
std,1.544638e+03,1.544636e+05,1.378709e+03,1.545144e+03,1.545143e+05,1.378273e+03,9.187862e-03,4.927751e+00,2.205487e+02,8.216434e+02,1.672517e+00,8.201554e+02,7.781802e+02,2.541717e-01
min,1.013500e+04,1.013506e+06,3.000700e+04,1.013500e+04,1.013506e+06,3.000700e+04,0.000000e+00,1.000000e+00,0.000000e+00,1.100000e+01,1.000000e+00,0.000000e+00,1.100000e+01,1.000000e+00
25%,1.129800e+04,1.129806e+06,3.072100e+04,1.129800e+04,1.129806e+06,3.072100e+04,0.000000e+00,1.000000e+00,1.475000e+02,7.170000e+02,2.000000e+00,7.130000e+02,6.590000e+02,2.000000e+00
50%,1.291500e+04,1.291503e+06,3.170300e+04,1.290200e+04,1.290205e+06,3.170300e+04,0.000000e+00,1.000000e+00,2.285000e+02,1.090000e+03,3.000000e+00,1.089000e+03,1.023000e+03,2.000000e+00
75%,1.410700e+04,1.410702e+06,3.310500e+04,1.410700e+04,1.410702e+06,3.310500e+04,0.000000e+00,1.000000e+00,3.370000e+02,1.759000e+03,4.000000e+00,1.754000e+03,1.660000e+03,2.000000e+00
max,1.686900e+04,1.686902e+06,3.642200e+04,1.686900e+04,1.686902e+06,3.642200e+04,1.000000e+00,1.032000e+03,3.873500e+04,1.134400e+04,2.300000e+01,1.134400e+04,9.411000e+03,2.000000e+00


In [6]:
print("\n===Missing Value Count===")
print(df.isna().sum())


===Missing Value Count===
ORIGIN_AIRPORT_ID              0
ORIGIN_AIRPORT_SEQ_ID          0
ORIGIN_CITY_MARKET_ID          0
DEST_AIRPORT_ID                0
DEST_AIRPORT_SEQ_ID            0
DEST_CITY_MARKET_ID            0
REPORTING_CARRIER        8297869
TICKET_CARRIER           8297869
OPERATING_CARRIER        8297869
BULK_FARE                8297869
PASSENGERS               8297869
MARKET_FARE              8297869
MARKET_DISTANCE          8297869
DISTANCE_GROUP           8297869
MARKET_MILES_FLOWN       8297869
NONSTOP_MILES            8297869
MKT_GEO_TYPE             8297869
dtype: int64


In [7]:
# Missing Value Example
df[df['REPORTING_CARRIER'].isna()].head()

,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,DISTANCE_GROUP,MARKET_MILES_FLOWN,NONSTOP_MILES,MKT_GEO_TYPE
24045317,10135,1013506,30135,10136,1013603,30136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24045318,10135,1013506,30135,10140,1014005,30140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24045319,10135,1013506,30135,10140,1014005,30140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24045320,10135,1013506,30135,10140,1014005,30140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24045321,10135,1013506,30135,10140,1014005,30140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
missing = df[df['REPORTING_CARRIER'].isna()]
print(missing['ORIGIN_AIRPORT_ID'].nunique(), "unique origin airports")
print(missing['ORIGIN_AIRPORT_ID'].value_counts().head(10))

447 unique origin airports
ORIGIN_AIRPORT_ID
11292    239079
12892    227918
13930    214651
12889    203318
10397    186739
14747    183678
13204    181860
11298    177599
14107    174452
14771    171039
Name: count, dtype: int64


realized a lot of US airports have missing info

In [9]:
origin_dest_pairs = missing.groupby(['ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID']).size().reset_index(name='count')
origin_dest_pairs.sort_values(by='count', ascending=False, inplace=True)
origin_dest_pairs.head()

,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,count
34463,12478,12892,7630
36394,12892,12478,7526
36328,12892,11618,7492
22027,11618,12892,7385
8759,10721,14771,7097


highest number of missing values for 
10721	Boston, MA: Logan International
11618	Newark, NJ: Newark Liberty International
12478	New York, NY: John F. Kennedy International
12892	Los Angeles, CA: Los Angeles International
14771	San Francisco, CA: San Francisco International


In [10]:
df_no_na = df.dropna()

In [11]:
df_no_na.info()
df_no_na.describe()
print("\n===Missing Value Count===")
print(df_no_na.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 24045317 entries, 0 to 24045316
Data columns (total 17 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ORIGIN_AIRPORT_ID      int64  
 1   ORIGIN_AIRPORT_SEQ_ID  int64  
 2   ORIGIN_CITY_MARKET_ID  int64  
 3   DEST_AIRPORT_ID        int64  
 4   DEST_AIRPORT_SEQ_ID    int64  
 5   DEST_CITY_MARKET_ID    int64  
 6   REPORTING_CARRIER      object 
 7   TICKET_CARRIER         object 
 8   OPERATING_CARRIER      object 
 9   BULK_FARE              float64
 10  PASSENGERS             float64
 11  MARKET_FARE            float64
 12  MARKET_DISTANCE        float64
 13  DISTANCE_GROUP         float64
 14  MARKET_MILES_FLOWN     float64
 15  NONSTOP_MILES          float64
 16  MKT_GEO_TYPE           float64
dtypes: float64(8), int64(6), object(3)
memory usage: 3.2+ GB

===Missing Value Count===
ORIGIN_AIRPORT_ID        0
ORIGIN_AIRPORT_SEQ_ID    0
ORIGIN_CITY_MARKET_ID    0
DEST_AIRPORT_ID          0
DEST_AIRP

In [14]:
# Save the cleaned DataFrame to a new CSV file
df_no_na.to_csv('../../data/clean_data/T_DB1B_MARKET_CLEAN.csv', index=False)